# 05 · Shared common: dates and hashing

This notebook exercises `common/dates.py` (canonical dates, vintage buckets, date synonyms, contiguous runs, the SQL date predicates) and `common/hashing.py` (key normalisation, `hash32`, fraction / fixed sampling) on **real data** from one of your pairs.
It mirrors `tests/unit/common/test_dates.py` and `tests/unit/common/test_hashing.py`. `src/dtrack_left/common` and `src/dtrack_right/common` are byte-identical copies, so every function is run through **both** copies and the results are compared.
Nothing is written anywhere except a local work folder.

## How to approach

- **Prerequisites**: kernel = the repo `.venv` (`uv sync --extra dbx`, plus `--extra athena` / `--extra ldap` if the pair needs them). The repo `.env` holds the Left credentials (`<macro>_USR`, `<macro>_PWD`, `<macro>_DSN`, or `ORACLE_*`). `~/.databrickscfg` has the `DBX_PROFILE` profile; `DTRACK_DBX_WAREHOUSE_ID` names a running SQL warehouse.
- **Run order**: top to bottom. Setup + counts are quick (one GROUP BY per side). The one-day pull is the only slow cell; it picks the smallest common day and stops above `MAX_ROWS`. Every later section only reads the two local CSVs, except section 4 which runs two COUNT(*) queries.
- **What to look at**: raw date values vs their canonical form on each side, the SQL the predicates generate for your real `date_type`, whether fraction sampling keeps the *same keys* on both sides, and the final "both copies agree" table (all zeros).
- **Failure modes**: `preflight` not ok = credentials / table name / missing extra; `unsupported date value` = a date format `canonical_date` cannot read (a real finding); no common day = `date_col` / `date_type` / `where` differ between the sides; warehouse errors with `external_links` = presigned downloads blocked, use `DISPOSITION = "inline"`.

In [ ]:
import os
from pathlib import Path

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")      # your real pairs config
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")                               # one active pair in it
TO_DATE = None                                                                    # None = yesterday; or "YYYY-MM-DD"
DAYS = 3                                                                          # window = DAYS days ending at TO_DATE
DBX_PROFILE = os.environ.get("DTRACK_DBX_PROFILE", "corp")                        # ~/.databrickscfg profile
WAREHOUSE_ID = os.environ.get("DTRACK_DBX_WAREHOUSE_ID", "")                      # SQL warehouse id for the Right side
DISPOSITION = os.environ.get("DTRACK_LOCAL_DISPOSITION", "external_links")        # "inline" when presigned downloads are blocked
ENV_FILE = os.environ.get("DTRACK_ENV_FILE")                                      # None = <repo>/.env
MAX_ROWS = 200_000                                                                # refuse the one-day pull above this
KEY_COLUMNS = None                                                                # None = the pair's key; or ["ORDER_ID"]
SAMPLE_RATE = 0.25                                                                # fraction for the sampling demo
SAMPLE_COUNT = 20                                                                 # fixed-N for the sampling demo
WORKDIR = Path.home() / ".local/plans/nbs/_work/05"                               # local scratch folder
KEEP = False                                                                      # keep WORKDIR after the run

In [ ]:
# Imports used across the notebook
import json
import logging
import shutil
from datetime import date, timedelta
from types import SimpleNamespace

import pandas as pd
from dotenv import load_dotenv

import dtrack_left
from dtrack_left.backends.databricks import workspace
from dtrack_left.backends.sources import source_from_config
from dtrack_left.cli.config import load_config
from dtrack_local.pipeline import WarehouseTarget
from dtrack_local.warehouse import Warehouse

# The repo root (editable install: <repo>/src/dtrack_left/__init__.py) and its .env with the
# Left credentials (<macro>_USR / _PWD / _DSN, ORACLE_*, LDAP_DSN, DTRACK_ORACLE_MACROS ...)
REPO = Path(dtrack_left.__file__).resolve().parents[2]
env_file = Path(ENV_FILE) if ENV_FILE else REPO / ".env"
loaded = load_dotenv(env_file, override=False)
print("repo :", REPO)
print(".env :", env_file, "(loaded)" if loaded else "(not found - credentials must already be in the environment)")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

# The real pairs config, validated exactly like `dtrack row-check` does
config = load_config(CONFIG_PATH)
active = [name for name, spec in config["pairs"].items() if not spec.get("skip")]
print("active pairs:", active)
if PAIR not in active:
    raise KeyError(f"PAIR={PAIR!r} is not an active pair in {CONFIG_PATH}")

# The pair, deep-copied, with its date range narrowed to the check window (TO_DATE back DAYS days)
to_date = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
from_date = (date.fromisoformat(to_date) - timedelta(days=DAYS - 1)).isoformat()
pair = json.loads(json.dumps(config["pairs"][PAIR]))
pair["fromDate"], pair["toDate"] = from_date, to_date
print(f"pair {PAIR}: window {from_date} .. {to_date}")
print("  left :", {k: v for k, v in pair["left"].items() if k not in ("user", "password", "dsn")})
print("  right:", pair["right"])

WORKDIR.mkdir(parents=True, exist_ok=True)

# Both copies of the shared common package, side by side
from dtrack_left.common import dates as left_dates, hashing as left_hashing
from dtrack_right.common import dates as right_dates, hashing as right_hashing

L = SimpleNamespace(dates=left_dates, hashing=left_hashing)
R = SimpleNamespace(dates=right_dates, hashing=right_hashing)
COPIES = {"dtrack_left": L, "dtrack_right": R}

## Connect and count

Preflight both sides, then count rows per day on each side for the window. These counts are the real input to the date functions below.

In [ ]:
# The real Left source (Oracle / Athena / file / duckdb) and the real Right side through the SQL warehouse
source = source_from_config(pair["left"])
warehouse = Warehouse(workspace(DBX_PROFILE), WAREHOUSE_ID, disposition=DISPOSITION)
right = WarehouseTarget(warehouse, pair["right"], WORKDIR / "right")

# Preflight both sides before any counting: Left SELECT ... WHERE 1=0, Right DESCRIBE QUERY
left_health = source.preflight()
right_health = right.preflight()
print("left  preflight:", left_health)
print("right preflight:", {k: v for k, v in right_health.items() if k != "columns"})
if not (left_health["ok"] and right_health["ok"]):
    raise RuntimeError("fix the preflight error above before going on")

In [ ]:
# Exact per-day counts on both sides for the window (the Stage 1 row-check numbers)
left_counts = source.partition_counts(from_date, to_date)
right_counts = right.partition_counts(from_date, to_date)
both = sorted(set(left_counts) & set(right_counts))
display(pd.DataFrame(
    [{"day": d, "left": left_counts.get(d), "right": right_counts.get(d)}
     for d in sorted(set(left_counts) | set(right_counts))]
))
if not both:
    raise RuntimeError("no day present on both sides: check date_col / date_type / where, or widen DAYS")

## 1 · canonical_date on real date values

Every partition key goes through `canonical_date` (ISO `YYYY-MM-DD`). Below are the raw day values exactly as each driver returns them (Python `datetime` from Oracle, text from the warehouse), then their canonical form from both copies.

In [ ]:
from dtrack_left.common.dates import bounds_where, day_expr

# Raw distinct day values from the Left driver, using the same day expression partition_counts uses
left_ref = day_expr(pair["left"]["date_col"], source.dialect, pair["left"].get("date_type"))
left_bounds = bounds_where(pair["left"]["date_col"], dialect=source.dialect,
                           date_type=pair["left"].get("date_type"), from_date=from_date, to_date=to_date)
_cols, left_raw = source.query(f"SELECT DISTINCT {left_ref} FROM {source.relation()} WHERE {source._where(left_bounds)}")

# Raw distinct day values from the warehouse (JSON_ARRAY results arrive as text)
right_ref = day_expr(pair["right"]["date_col"], "spark", pair["right"].get("date_type"))
right_bounds = bounds_where(pair["right"]["date_col"], dialect="spark",
                            date_type=pair["right"].get("date_type"), from_date=from_date, to_date=to_date)
right_raw = warehouse.rows(f"SELECT DISTINCT {right_ref} FROM {right.relation()} WHERE {right._where(right_bounds)}")

# canonical_date through both copies; an unreadable value shows its error instead of stopping the notebook
def canon(copy, value):
    try:
        return copy.dates.canonical_date(value)
    except ValueError as exc:
        return f"ERROR {exc}"

rows = [{"side": side, "raw": repr(value), "type": type(value).__name__,
         "left copy": canon(L, value), "right copy": canon(R, value)}
        for side, raw in (("left", left_raw), ("right", right_raw)) for (value,) in raw]
display(pd.DataFrame(rows).sort_values(["side", "left copy"]))

In [ ]:
# The date column as it lands in the pulled CSVs is text: show what canonical_date makes of it
# (needs the one-day pull from section 5; run this cell again after it if you skipped ahead)
if "left_frame" in globals():
    for side, frame, column in (("left", left_frame, pair["left"]["date_col"]),
                                ("right", right_frame, pair["right"]["date_col"])):
        sample = frame[column].dropna().drop_duplicates().head(3).tolist()
        print(side, [(v, canon(L, v)) for v in sample])
else:
    print("run section 5 (the one-day pull) first, then come back to this cell")

In [ ]:
# Illustrative literals (NOT from your data): every accepted input format, and one rejected value
examples = ["2026-01-02", " 20260102 ", "01/02/2026", "02JAN2026", "02-Jan-2026", "next tuesday"]
display(pd.DataFrame([{"input": repr(v), "canonical": canon(L, v)} for v in examples]))

## 2 · bucket_date: real days into vintages

Column statistics are compared per *window*: the days of one vintage bucket (`day`, `week`, `month`, `quarter`, `year`). The pair's configured vintage is marked.

In [ ]:
# Each real Left day floored to every vintage; the pair's vintage decides the Stage 2 windows
VINTAGES = ["day", "week", "month", "quarter", "year"]
print("pair vintage:", pair.get("vintage") or "day (default)")
display(pd.DataFrame([{"day": d, **{v: L.dates.bucket_date(d, v) for v in VINTAGES}} for d in sorted(left_counts)]))

# Rows per bucket for the pair's own vintage (what one Stage 2 window would cover)
vintage = pair.get("vintage") or "day"
per_bucket = {}
for d, n in left_counts.items():
    bucket = L.dates.bucket_date(d, vintage)
    per_bucket[bucket] = per_bucket.get(bucket, 0) + n
print(f"left rows per {vintage} bucket:", dict(sorted(per_bucket.items())))

# An unknown vintage is refused (illustrative)
try:
    L.dates.bucket_date(to_date, "fortnight")
except ValueError as exc:
    print("refused:", exc)

## 3 · apply_synonyms and contiguous_runs

`apply_synonyms` folds raw partition keys to canonical days and maps sentinel dates through the pair's `date_synonyms`. `contiguous_runs` turns a set of days into ranges, splitting on holes; the predicates in section 4 are built from these runs.

In [ ]:
# The pair's date_synonyms (empty for most pairs) applied to the real counts of each side
synonyms = pair.get("date_synonyms") or {}
print("date_synonyms:", synonyms or "(none)")
folded_left = L.dates.apply_synonyms(left_counts, synonyms)
folded_right = R.dates.apply_synonyms(right_counts, synonyms)
print("left  folded:", folded_left)
print("right folded:", folded_right)

# Runs of consecutive days per side; a hole (a day with no rows) splits a run
print("left  runs:", L.dates.contiguous_runs(list(left_counts)))
print("right runs:", R.dates.contiguous_runs(list(right_counts)))
window_days = [(date.fromisoformat(from_date) + timedelta(days=i)).isoformat() for i in range(DAYS)]
print("days in the window with no Left rows :", [d for d in window_days if d not in left_counts])
print("days in the window with no Right rows:", [d for d in window_days if d not in right_counts])

## 4 · The SQL date predicates for this pair

`quote_ident`, `day_expr`, `date_literal`, `bounds_where` and `partition_where` build every date filter dtrack sends. Here they are rendered for your pair's real dialect and `date_type` on each side, then one predicate with a deliberate hole is executed on both sides and checked against the counts.

In [ ]:
# The building blocks for the pair's real columns, Left dialect and Spark (Right)
sides = {
    "left": (pair["left"]["date_col"], source.dialect, pair["left"].get("date_type")),
    "right": (pair["right"]["date_col"], "spark", pair["right"].get("date_type")),
}
for side, (column, dialect, date_type) in sides.items():
    print(f"--- {side}: column={column!r} dialect={dialect} date_type={date_type!r}")
    print("quote_ident  :", L.dates.quote_ident(column, dialect))
    print("day_expr     :", L.dates.day_expr(column, dialect, date_type))
    print("date_literal :", L.dates.date_literal(to_date, dialect, date_type))
    print("bounds_where :", L.dates.bounds_where(column, dialect=dialect, date_type=date_type, from_date=from_date, to_date=to_date))
    print("partition_where(all days):", L.dates.partition_where(column, sorted(left_counts), dialect=dialect, date_type=date_type))
print("partition_where([]) ->", L.dates.partition_where("dt", [], dialect="spark"))

In [ ]:
# A real day list with a hole (the middle day dropped when there are 3+), as a union of ranges
chosen = sorted(both)
if len(chosen) >= 3:
    chosen = [chosen[0]] + chosen[2:]
expected = sum(left_counts[d] for d in chosen), sum(right_counts[d] for d in chosen)
left_pred = L.dates.partition_where(pair["left"]["date_col"], chosen, dialect=source.dialect, date_type=pair["left"].get("date_type"))
right_pred = R.dates.partition_where(pair["right"]["date_col"], chosen, dialect="spark", date_type=pair["right"].get("date_type"))
print("days :", chosen)
print("left :", left_pred)
print("right:", right_pred)

# Execute the predicates (read-only COUNT(*)) and compare with the per-day counts above
_c, got_left = source.query(f"SELECT COUNT(*) FROM {source.relation()} WHERE {source._where(left_pred)}")
got_right = warehouse.rows(f"SELECT COUNT(*) FROM {right.relation()} WHERE {right._where(right_pred)}")
print(f"left  COUNT(*) = {int(got_left[0][0])}  (sum of day counts {expected[0]})")
print(f"right COUNT(*) = {int(got_right[0][0])}  (sum of day counts {expected[1]})")

## 5 · Pull one real day from both sides

The smallest day present on both sides is pulled exactly like the col-check pulls it (Left `pull_window`, Right `window_sql` streamed through the warehouse). Sections 6-8 only use these two CSVs.

In [ ]:
# Column names: Left value columns (date_col excluded), keys, their Right names and key families
date_col = pair["left"]["date_col"]
value_cols = [c for c in pair["col_map"] if c.lower() != date_col.lower()]
keys = list(KEY_COLUMNS or [k.strip() for k in str(pair.get("key") or "").split(",") if k.strip()])
if not keys:
    keys = value_cols[:1]
    print(f"pair has no key; using {keys} as the sampling key for this demo (set KEY_COLUMNS to choose)")
right_keys = [pair["col_map"].get(k, k) for k in keys]
families = [dict(pair.get("key_families") or {}).get(k, "string") for k in keys]
print("value columns:", value_cols)
print("keys:", keys, "-> right", right_keys, "families", families)

# The smallest day present on both sides, so the pull stays small (MAX_ROWS guards it)
DAY = min(both, key=lambda d: left_counts[d])
if left_counts[DAY] > MAX_ROWS:
    raise RuntimeError(f"{DAY} has {left_counts[DAY]} Left rows (> MAX_ROWS); raise MAX_ROWS or pick another pair")
print(f"pulling {DAY}: left={left_counts[DAY]} right={right_counts[DAY]} rows")

# Left: the same pull_window the col-check uses (date_col + keys + value columns, raw text CSV)
left_csv = source.pull_window([DAY], value_cols, WORKDIR / "left.csv", key_columns=keys)

# Right: the same window SQL WarehouseTarget builds (every column CAST AS STRING), streamed to CSV.
# window_sql only reads col_map / columns / right_key_cols / sample from the manifest; we pass just those.
mini_manifest = {"col_map": pair["col_map"], "columns": value_cols, "right_key_cols": right_keys, "sample": {"mode": "all"}}
right_sql, right_selected = right.window_sql(mini_manifest, {"partitions": [DAY]})
print("right SQL:", right_sql)
right_rows = warehouse.to_csv(right_sql, WORKDIR / "right.csv", right_selected)
right_csv = WORKDIR / "right.csv"

left_frame = pd.read_csv(left_csv, dtype=object)
right_frame = pd.read_csv(right_csv, dtype=object)
print(f"left.csv  {len(left_frame)} rows  {list(left_frame.columns)}")
print(f"right.csv {len(right_frame)} rows  {list(right_frame.columns)}")
display(left_frame.head(3))
display(right_frame.head(3))

## 6 · Key normalisation on real keys

Sampling hashes a *normalised* key so both sides pick the same rows: `<<NULL>>` for blanks, `canonical_date` for the `date` family, `collapse_decimal` for `numeric` (only pure decimals lose trailing zeros; leading zeros and exponents stay), lower-cased NFC text for `string`.

In [ ]:
# Raw key text vs normalised form, Left and Right, for a few real keys
def norm(copy, row, cols):
    return tuple(copy.hashing.normalize_value(row[c] if pd.notna(row[c]) else None, f) for c, f in zip(cols, families))

show = []
for side, frame, cols in (("left", left_frame, keys), ("right", right_frame, right_keys)):
    for _, row in frame.head(5).iterrows():
        show.append({"side": side, "raw": tuple(row[c] for c in cols), "normalised": norm(L, row, cols)})
display(pd.DataFrame(show))

# Where collapse_decimal actually changes a real value (numeric-looking text in any value column)
changed = []
for side, frame in (("left", left_frame), ("right", right_frame)):
    for column in frame.columns:
        for value in frame[column].dropna().drop_duplicates().head(2000):
            text = str(value).strip()
            if L.hashing.collapse_decimal(text) != text:
                changed.append({"side": side, "column": column, "raw": text, "collapsed": L.hashing.collapse_decimal(text)})
                break
display(pd.DataFrame(changed) if changed else "no value in this day needs decimal collapsing")

## 7 · hash32 and fraction / fixed sampling on both sides

`hash32` is the first 8 hex digits of MD5 of the normalised composite key (`a|b|c`); a fraction sample keeps a row when `hash32 < round(rate * 2**32)`. Because both sides hash the same normalised text, the kept key sets must agree wherever the key exists on both sides.

In [ ]:
import hashlib

# hash32 of a few real composite keys; it equals int(md5[:8], 16) and stays below 2**32
threshold = round(SAMPLE_RATE * L.hashing.HASH_BUCKET_SPACE)
for _, row in left_frame.head(5).iterrows():
    composite = "|".join(norm(L, row, keys))
    h = L.hashing.hash32(composite)
    print(f"{composite!r:40} hash32={h:>10}  md5[:8]={int(hashlib.md5(composite.encode()).hexdigest()[:8], 16):>10}"
          f"  selected@{SAMPLE_RATE}={L.hashing.selected_fraction([row[c] for c in keys], families, threshold)}")

In [ ]:
# sample_csv in every mode on the real CSVs (outputs go next to them as *.sampled.csv)
kept = {}
for side, path, cols in (("left", left_csv, keys), ("right", right_csv, right_keys)):
    same = L.hashing.sample_csv(path, key_columns=cols, mode="all")
    frac = pd.read_csv(L.hashing.sample_csv(path, key_columns=cols, mode="fraction", rate=SAMPLE_RATE, families=families), dtype=object)
    again = pd.read_csv(L.hashing.sample_csv(path, key_columns=cols, mode="fraction", rate=SAMPLE_RATE, families=families), dtype=object)
    fixed = pd.read_csv(L.hashing.sample_csv(path, key_columns=cols, mode="fixed", count=SAMPLE_COUNT, families=families), dtype=object)
    kept[side] = {tuple(norm(L, row, cols)) for _, row in frac.iterrows()}
    total = len(pd.read_csv(path, dtype=object))
    print(f"{side:5}: all -> {same.name} (input untouched) | fraction {len(frac)}/{total} rows"
          f" (deterministic: {frac.equals(again)}) | fixed {SAMPLE_COUNT} keys -> {len(fixed)} rows, {len(fixed[cols].drop_duplicates())} distinct keys")

# Parity: among keys present on both sides, the fraction sample keeps exactly the same ones
left_keys_all = {tuple(norm(L, row, keys)) for _, row in left_frame.iterrows()}
right_keys_all = {tuple(norm(L, row, right_keys)) for _, row in right_frame.iterrows()}
shared = left_keys_all & right_keys_all
print(f"keys on both sides: {len(shared)} | kept left {len(kept['left'] & shared)} | kept right {len(kept['right'] & shared)}"
      f" | disagreements: {len((kept['left'] ^ kept['right']) & shared)}")

In [ ]:
# Fixed-N with key tuples: the Left side picks N keys once, the Right side keeps rows matching those tuples
left_fixed = pd.read_csv(L.hashing.sample_csv(left_csv, key_columns=keys, mode="fixed", count=SAMPLE_COUNT, families=families), dtype=object)
tuples = left_fixed[keys].drop_duplicates().values.tolist()
right_fixed = pd.read_csv(R.hashing.sample_csv(right_csv, key_columns=right_keys, mode="fixed", count=SAMPLE_COUNT,
                                               key_tuples=tuples, families=families), dtype=object)
print(f"left picked {len(tuples)} keys ({len(left_fixed)} rows); right matched {len(right_fixed[right_keys].drop_duplicates())} of them ({len(right_fixed)} rows)")

# Guard rails (illustrative): sampling without keys, and an unknown mode
for kwargs in ({"key_columns": [], "mode": "fraction", "rate": 0.5}, {"key_columns": keys, "mode": "stratified"}):
    try:
        L.hashing.sample_csv(left_csv, **kwargs)
    except ValueError as exc:
        print("refused:", exc)

## 8 · Both copies agree

The Left package and the Databricks job each ship their own `common/`. They must be byte-identical and give identical answers; this runs every date and hashing function through both copies over all the real values seen above.

In [ ]:
# Byte-identical source files in the two installed copies
left_dir, right_dir = Path(left_dates.__file__).parent, Path(right_dates.__file__).parent
for name in sorted(p.name for p in left_dir.glob("*.py")):
    same = (left_dir / name).read_bytes() == (right_dir / name).read_bytes()
    print(f"{name:15} {'identical' if same else 'DRIFTED'}")

In [ ]:
# Every function, both copies, over the real values; each row counts disagreements (expect 0)
raw_days = [v for (v,) in left_raw] + [v for (v,) in right_raw]
all_days = sorted(set(left_counts) | set(right_counts))
all_text = [v for frame in (left_frame, right_frame) for c in frame.columns for v in frame[c].dropna().drop_duplicates().head(500)]

def disagreements(fn, values):
    return sum(1 for v in values if repr(fn(L, v)) != repr(fn(R, v)))

def safe(f):
    def run(copy, v):
        try:
            return f(copy, v)
        except Exception as exc:
            return type(exc).__name__
    return run

checks = {
    "canonical_date(raw days)": (safe(lambda c, v: c.dates.canonical_date(v)), raw_days),
    "bucket_date x 5 vintages": (lambda c, v: [c.dates.bucket_date(v, x) for x in VINTAGES], all_days),
    "apply_synonyms(counts)": (lambda c, v: c.dates.apply_synonyms(v, synonyms), [left_counts, right_counts]),
    "contiguous_runs(days)": (lambda c, v: c.dates.contiguous_runs(v), [list(left_counts), list(right_counts)]),
    "partition_where(left)": (lambda c, v: c.dates.partition_where(pair["left"]["date_col"], v, dialect=source.dialect, date_type=pair["left"].get("date_type")), [all_days, chosen]),
    "partition_where(right)": (lambda c, v: c.dates.partition_where(pair["right"]["date_col"], v, dialect="spark", date_type=pair["right"].get("date_type")), [all_days, chosen]),
    "collapse_decimal(text)": (lambda c, v: c.hashing.collapse_decimal(str(v).strip()), all_text),
    "normalize_value string/numeric/date": (lambda c, v: [safe(lambda c2, v2: c2.hashing.normalize_value(v2, f))(c, v) for f in ("string", "numeric", "date")], all_text),
    "hash32(text)": (lambda c, v: c.hashing.hash32(str(v)), all_text),
}
display(pd.DataFrame([{"check": name, "values": len(values), "disagreements": disagreements(fn, values)}
                      for name, (fn, values) in checks.items()]))

# The sampled files themselves: same rows from both copies
a = pd.read_csv(L.hashing.sample_csv(left_csv, key_columns=keys, mode="fraction", rate=SAMPLE_RATE, families=families), dtype=object)
b = pd.read_csv(R.hashing.sample_csv(left_csv, key_columns=keys, mode="fraction", rate=SAMPLE_RATE, families=families), dtype=object)
print("fraction sample of left.csv identical across copies:", a.equals(b))

## Cleanup

Only the local `WORKDIR` was written (`left.csv`, `right.csv` and their `*.sampled.csv`).

In [ ]:
# Remove the local working folder unless KEEP=True (nothing else was written anywhere)
if KEEP:
    print("kept", WORKDIR)
else:
    shutil.rmtree(WORKDIR, ignore_errors=True)
    print("removed", WORKDIR)

## What to check

- Both preflights ok, and at least one day present on both sides.
- Section 1: every raw day value has a canonical form (no `ERROR`); the CSV date text may need a time-free format.
- Section 4: each executed `COUNT(*)` equals the sum of the per-day counts (the predicates select exactly those days, holes included).
- Section 7: fraction sampling is deterministic and has **0 disagreements** on keys shared by both sides; fixed-N keys chosen on the Left are found on the Right.
- Section 8: all common files `identical`, every disagreement count `0`.